# Fraud Random Forest with 7-Day Category Share (F1 0.9749) - Activity FLAML (Macro-F1 0.6451)

Fraud F1 of **0.9749** was measured in the follow-up experiment on 140,438 later transactions. The forest used the preceding seven days’ category share and its change from the previous 28 days. That experiment ran separately from the code below. Activity macro-F1 of **0.6451** is the pooled four-fold result for the macro-F1-tuned FLAML search in this notebook.


# AutoML versus the current competition models
Run all cells on a CPU runtime. Upload cct_train.zip and har_train.csv when prompted. This standalone copy uses the same features, chronological windows, four duplicate-group activity folds and current selected models as the combined-features notebook.

FLAML searches four tree-based learner families separately for accuracy and class-specific fraud F1 or activity macro-F1. Each search allows up to 24 trials or 90 seconds. For speed, each inner fraud fit uses a seeded, prevalence-preserving sample of up to 60,000 earlier rows. The validation rows, threshold-calibration fits and outer model fits use their full assigned data. Time-limited results can vary with hardware. These are development comparisons on course training data. See [FLAML's task-oriented AutoML guide](https://microsoft.github.io/FLAML/docs/Use-Cases/Task-Oriented-AutoML/).

### Measured results
The four-fold activity comparison favored the macro-F1 AutoML search (78.58% accuracy, 0.6451 macro-F1). The later fraud comparison favored the current forest (99.8911% accuracy, 0.8346 fraud F1, 0.8493 average precision at the previous operating point). The original combined-features experiments and their outputs are included below. The new searches start at **Search settings and metrics**. Each section uses its own export directory.

# Combined features and ensembles
In Colab, select **Runtime → Run all** on a CPU runtime. Upload the supplied **cct_train.zip** and **har_train.csv** when prompted. Keep the course data and trained artifacts private. Locally, run from the repository or its Code Prototypes directory.

Each task ranks candidates by the exact number of correct held-out predictions. F1 breaks exact accuracy ties. The fraud task reserves a later period for a final check. The activity leaderboard is a cross-validation development estimate. The last cell refits both winners and offers a private model download.

### Results from this run
The selected fraud forest combined merchant history, card spending trend and time-of-day spending comparisons with a minimum leaf size of 2. It made 15 more correct predictions on 140,438 later transactions than the previous development winner. At its accuracy-selected threshold it reached 99.8911% accuracy and 0.8346 fraud F1. The highest later-period fraud F1 among the metrics below was 0.8604 for both histories at leaf size 4 with a development-F1-selected threshold. This period has been reviewed in earlier rounds.

For activity recognition, forest/SVM/logistic probability weights of 4:2:1 yielded 77.68% accuracy and 0.6077 macro-F1 across four grouped folds. The previous forest/SVM blend scored 77.11% and 0.5714. Appending principal components to the forest gave the three-model blend 0.6109 macro-F1 and 77.56% accuracy.

The export uses development accuracy and breaks ties with F1. Thresholds for both fraud objectives and the per-fold activity comparisons appear below.

In [1]:
from pathlib import Path
from collections import deque
import zipfile
import shutil
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 610
import os
WORKERS = min(8, os.cpu_count() or 1)
print("scikit-learn", sklearn.__version__)
def locate(name):
    for root in (Path.cwd(), Path.cwd().parent):
        direct = root / name
        if direct.is_file():
            return direct
        data = root / "Docs and Data"
        if data.is_dir():
            match = next(data.rglob(name), None)
            if match:
                return match

if not all(locate(n) for n in ("cct_train.zip", "har_train.csv")):
    from google.colab import files
    files.upload()
fraud_path, har_path = map(locate, ("cct_train.zip", "har_train.csv"))
if not fraud_path or not har_path:
    raise FileNotFoundError("Upload both cct_train.zip and har_train.csv using their original names.")
entity_fields = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
event_fields = entity_fields + ["unix_time", "trans_time", "trans_date", "amt", "lat", "long", "merch_lat", "merch_long"]
with zipfile.ZipFile(fraud_path) as archive:
    member, = [n for n in archive.namelist() if Path(n).name == "cct_train.csv"]
    with archive.open(member) as stream:
        fraud = pd.read_csv(stream, usecols=event_fields + ["is_fraud"], dtype={k:"string" for k in entity_fields})
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)
har = pd.read_csv(har_path, header=None)
print("Fraud rows:", len(fraud), "fraud rate:", fraud.is_fraud.mean())
print("Activity rows:", len(har), "predictors:", har.shape[1]-1)

scikit-learn 1.9.1
Fraud rows: 700807 fraud rate: 0.005106969536548579
Activity rows: 2442 predictors: 328


## Fraud · compare a purchase with earlier purchases
Compare the same tree on three increasingly rich feature sets. Card features describe spending and activity in the previous hour and day. The widest set adds account and purchaser history, familiarity with this merchant, category peers and address-to-merchant distance. First name, last name and birth date form a purchaser proxy. The identity audit below shows whether that proxy actually links multiple cards. Shared names alone are ambiguous.

Counts and averages exclude the current transaction and all transactions with the same timestamp. Category peers include earlier purchases across cards. A new entity has zero prior transactions and an unknown average. Historical fraud labels never enter the feature builder. For the research basis see [Correa Bahnsen et al.](https://doi.org/10.1016/j.eswa.2015.12.030).

In [4]:
def fraud_features(events):
    # Return features in input order, including when called on a shuffled batch.
    d = events.reset_index(drop=True).sort_values("unix_time", kind="stable")
    x = pd.DataFrame(index=d.index)
    x["amount"] = d.amt
    x["hour"] = d.trans_time.str.slice(0, 2).astype(int)
    lat1, lat2 = np.radians(d.lat), np.radians(d.merch_lat)
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(np.radians(d.merch_long-d.long)/2)**2
    x["home_merchant_km"] = 12742 * np.arcsin(np.sqrt(a.clip(0, 1)))

    d["time_band"] = x["hour"] // 6
    entities = {"card":["cc_num"], "card_category":["cc_num","category"],
                "account":["acct_num"], "person":["first","last","dob"],
                "card_merchant":["cc_num","merchant"], "category":["category"], "merchant":["merchant"],
                "merchant_time":["merchant","time_band"], "card_time":["cc_num","time_band"]}
    for prefix, keys in entities.items():
        # Collapse simultaneous events so their within-second ordering is immaterial.
        group_keys = keys + ["unix_time"]
        present = d[keys].notna().all(axis=1)
        batch = (d.loc[present].groupby(group_keys, sort=False, observed=True)
                 .amt.agg(total="sum", n="size").reset_index())
        groups = batch.groupby(keys, sort=False, observed=True)
        count = groups.n.cumsum() - batch.n
        total = groups.total.cumsum() - batch.total
        batch["count"] = count
        batch["mean"] = total / count.replace(0, np.nan)
        batch["elapsed"] = groups.unix_time.diff()
        prior = d[group_keys].merge(batch[group_keys + ["count","mean","elapsed"]],
                                    on=group_keys, how="left", sort=False, validate="many_to_one")
        prior.index = d.index
        x[prefix+"_count"] = prior["count"].fillna(0)
        x[prefix+"_amount_ratio"] = d.amt / prior["mean"].replace(0, np.nan)
        if prefix == "card":
            x["card_elapsed"] = prior["elapsed"]
    # Prefix sums and binary search make strict trailing windows inexpensive per card.
    for name in ("hour_count","day_count","day_spend"):
        x[name] = 0.
    x["reference_days_observed"] = 0.
    x["day_vs_prior28_ratio"] = np.nan
    for _, block in d.groupby("cc_num", sort=False):
        times = block.unix_time.to_numpy()
        end = times.searchsorted(times, side="left")
        hour = times.searchsorted(times-3600)
        day = times.searchsorted(times-86400)
        sums = np.r_[0., block.amt.to_numpy().cumsum()]
        x.loc[block.index, ["hour_count","day_count","day_spend"]] = np.column_stack(
            [end-hour, end-day, sums[end]-sums[day]])
        ref_start = times.searchsorted(times-29*86400, side="left")
        ref_total = sums[day]-sums[ref_start]
        coverage = np.clip((times-times[0])/86400-1, 0, 28)
        ratio = np.full(len(block), np.nan)
        np.divide(sums[end]-sums[day], ref_total/28, out=ratio,
                  where=(coverage == 28) & (ref_total > 0))
        x.loc[block.index, ["reference_days_observed","day_vs_prior28_ratio"]] = np.column_stack([coverage,ratio])
    return x.sort_index().astype("float32")

for keys in [["first","last"], ["first","last","dob"], ["acct_num"]]:
    card_counts = fraud.groupby(keys).cc_num.nunique()
    print(keys, "groups:", len(card_counts), "groups with multiple cards:", (card_counts > 1).sum())
Xf = fraud_features(fraud[event_fields])
yf = fraud.is_fraud.to_numpy(dtype=int)
basic = ["amount","hour"]
card = basic + ["card_amount_ratio","card_category_amount_ratio","hour_count","day_count","card_elapsed"]
extended = card + ["day_spend","card_count","card_merchant_count","home_merchant_km",
                   "account_count","account_amount_ratio","person_count","person_amount_ratio",
                   "category_amount_ratio"]
print("Feature sets:", len(basic), len(card), len(extended))

['first', 'last'] groups: 496 groups with multiple cards: 3
['first', 'last', 'dob'] groups: 500 groups with multiple cards: 0
['acct_num'] groups: 500 groups with multiple cards: 0
Feature sets: 2 7 16


### Development comparison
Start from the same 100-tree forest and compare merchant history, spending trend, and both together. For each combined feature set test minimum leaf sizes 2, 4, and 8. The extra context compares the purchase against this card at this merchant and spending in this six-hour time band at the merchant and on the card; historical counts indicate available evidence. Select by pooled development accuracy with fraud F1 breaking ties. Also evaluate a development-F1-tuned threshold.

In [5]:
def best_threshold(y, score, objective="accuracy"):
    order = np.argsort(-score, kind="stable")
    ranked, truth = score[order], y[order]
    ends = np.flatnonzero(np.r_[ranked[:-1] != ranked[1:], True])
    tp = np.r_[0, np.cumsum(truth)[ends]]
    positives = np.r_[0, ends+1]
    correct = (len(y)-y.sum()) + 2*tp - positives
    tied = np.flatnonzero(correct == correct.max())
    f1 = 2*tp / np.maximum(1, positives+y.sum())
    if objective == "fraud_f1":
        tied = np.flatnonzero(f1 == f1.max())
        winner = tied[np.argmax(correct[tied])]
    else:
        winner = tied[np.argmax(f1[tied])]
    return np.r_[np.inf, ranked[ends]][winner]

def fraud_metrics(y, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return dict(correct=int(tn+tp), accuracy=accuracy_score(y,pred), fraud_f1=f1_score(y,pred,zero_division=0),
                macro_f1=f1_score(y,pred,average="macro"), auc=roc_auc_score(y,score),
                false_alerts=fp, missed_frauds=fn, caught=tp)

dates = pd.to_datetime(fraud.trans_date)
edges = pd.to_datetime(["2022-03-01","2022-05-01","2022-07-01","2022-09-01"])
folds = [(np.flatnonzero(dates < a), np.flatnonzero((dates >= a) & (dates < b)))
         for a,b in zip(edges[:-1], edges[1:])]
def fit_fraud(name, rows):
    model, cols = fraud_models[name]
    return clone(model).fit(Xf.iloc[rows][cols], yf[rows]) if model is not None else None

def fraud_scores(model, frame, cols):
    if model is None:
        return frame["amount"].to_numpy()
    return model.predict_proba(frame[cols])[:,1]

from sklearn.impute import SimpleImputer
from itertools import product

def forest(leaf):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                         RandomForestClassifier(n_estimators=100, max_samples=.5,
                                                min_samples_leaf=leaf, n_jobs=WORKERS, random_state=SEED))

base_forest, merchant_forest, trend_forest = "Original", "Merchant history", "Card spending trend"
merchant = ["merchant_count", "merchant_amount_ratio"]
trend = ["reference_days_observed","day_vs_prior28_ratio"]
combined = extended + merchant + trend
context = combined + ["card_merchant_amount_ratio","card_category_count",
                      "merchant_time_count","merchant_time_amount_ratio","card_time_amount_ratio"]
fraud_models = {base_forest:(forest(4),extended), merchant_forest:(forest(4),extended+merchant),
                trend_forest:(forest(4),extended+trend)}
for feature_name, columns in [("Combined", combined), ("Combined + context", context)]:
    for leaf in (4,2,8):
        fraud_models[f"{feature_name}, leaf {leaf}"] = (forest(leaf),columns)
fraud_results = []
thresholds, f1_thresholds = {}, {}
for name, (_, cols) in fraud_models.items():
    print("Evaluating", name, flush=True)
    scores = np.concatenate([fraud_scores(fit_fraud(name,tr), Xf.iloc[va],cols) for tr,va in folds])
    truth = yf[np.concatenate([va for _,va in folds])]
    thresholds[name] = best_threshold(truth,scores)
    f1_thresholds[name] = best_threshold(truth,scores,"fraud_f1")
    fraud_results.append(dict(model=name, **fraud_metrics(truth,scores,thresholds[name])))
fraud_board = pd.DataFrame(fraud_results).sort_values(["correct","fraud_f1"],ascending=False,kind="stable")
display(fraud_board)
fraud_winner = fraud_board.iloc[0].model


Evaluating Original
Evaluating Merchant history
Evaluating Card spending trend
Evaluating Combined, leaf 4
Evaluating Combined, leaf 2
Evaluating Combined, leaf 8
Evaluating Combined + context, leaf 4
Evaluating Combined + context, leaf 2
Evaluating Combined + context, leaf 8
                        model  correct  accuracy  fraud_f1  macro_f1       auc  false_alerts  missed_frauds  caught
7  Combined + context, leaf 2   181898  0.998233  0.812573  0.905843  0.985917            88            234     698
2         Card spending trend   181872  0.998090  0.784920  0.891980  0.990157            51            297     635
6  Combined + context, leaf 4   181868  0.998068  0.792208  0.895619  0.988158            91            261     671
8  Combined + context, leaf 8   181866  0.998057  0.787515  0.893270  0.989060            78            276     656
0                    Original   181864  0.998046  0.780247  0.889633  0.991809            56            300     632
4            Combined, leaf

### Later-period checks
Compare the original forest, both previous challengers, the two histories together at leaf size 4, and the best development candidate from each combined family. This period has been inspected before. Selection uses the earlier windows.

In [6]:
# Compare the forests on the same later transactions.
# Both thresholds and the export winner were chosen before this comparison.
hold = np.flatnonzero(dates >= edges[-1])
before_hold = np.flatnonzero(dates < edges[-1])
hold_predictions, metrics, f1_metrics = {}, [], []
family_winners = [fraud_board.loc[fraud_board.model.str.startswith(prefix),"model"].iloc[0]
                  for prefix in ("Combined,","Combined + context,")]
for name in dict.fromkeys([base_forest,merchant_forest,trend_forest,"Combined, leaf 4",*family_winners,fraud_winner]):
    candidate_cols = fraud_models[name][1]
    scores = fraud_scores(fit_fraud(name, before_hold), Xf.iloc[hold], candidate_cols)
    threshold = thresholds[name]
    hold_predictions[name] = scores >= threshold
    f1_cut = f1_thresholds[name]
    f1_metrics.append(dict(model=name, threshold=f1_cut, **fraud_metrics(yf[hold],scores,f1_cut)))
    metrics.append(dict(model=name, threshold=threshold, **fraud_metrics(yf[hold], scores, threshold)))
print("Accuracy-selected development thresholds")
display(pd.DataFrame(metrics).set_index("model"))
print("Fraud-F1-selected development thresholds")
display(pd.DataFrame(f1_metrics).set_index("model"))
def paired_counts(truth, incumbent, challenger):
    was_right, now_right = incumbent == truth, challenger == truth
    fixed = int((~was_right & now_right).sum())
    lost = int((was_right & ~now_right).sum())
    return dict(rows=len(truth), fixed=fixed, lost=lost, net=fixed-lost,
                delta_pp=100*(fixed-lost)/len(truth))
def paired_by(truth, incumbent, challenger, slices):
    slices = np.asarray(slices)
    results = {"All": paired_counts(truth, incumbent, challenger)}
    for key in pd.unique(slices):
        mask = slices == key
        results[key] = paired_counts(truth[mask], incumbent[mask], challenger[mask])
    return pd.DataFrame.from_dict(results, orient="index")
display(paired_by(yf[hold], hold_predictions[base_forest], hold_predictions[merchant_forest],
                  dates.iloc[hold].dt.strftime("%Y-%m")))
print("Spending trend versus original forest")
display(paired_by(yf[hold], hold_predictions[base_forest], hold_predictions[trend_forest],
                  dates.iloc[hold].dt.strftime("%Y-%m")))
print("Selected model versus previous development winner")
display(paired_by(yf[hold],hold_predictions[trend_forest],hold_predictions[fraud_winner],dates.iloc[hold].dt.strftime("%Y-%m")))
cols = fraud_models[fraud_winner][1]
print("Development-selected fraud model:", fraud_winner)

Accuracy-selected development thresholds
                            threshold  correct  accuracy  fraud_f1  macro_f1       auc  false_alerts  missed_frauds  caught
model                                                                                                                      
Original                     0.297399   140272  0.998818  0.819172  0.909290  0.989406            35            131     376
Merchant history             0.341794   140304  0.999046  0.856838  0.928179  0.987966            28            106     401
Card spending trend          0.349480   140270  0.998804  0.816594  0.907997  0.982823            35            133     374
Combined, leaf 4             0.396183   140297  0.998996  0.846238  0.922867  0.984931            22            119     388
Combined, leaf 2             0.412000   140302  0.999032  0.852174  0.925844  0.989209            21            115     392
Combined + context, leaf 2   0.431333   140285  0.998911  0.834595  0.917024  0.980080     

## Activity recognition · compare linear, kernel, neighbor and tree models
Group identical feature vectors before stratification so a repeated sensor reading cannot appear in both training and validation. Fit scaling and constant-column removal inside each fold. The two logistic models measure the effect of class weighting on accuracy. The neighbor model tests whether proximity to similar sensor windows helps. See [Jordão et al. on activity-recognition evaluation](https://arxiv.org/abs/1806.05226).

In [2]:
Xh, yh = har.iloc[:,:-1], har.iloc[:,-1].to_numpy()
groups = pd.util.hash_pandas_object(Xh, index=False).to_numpy()
labels = np.unique(yh)
# Accept a split only if all activities occur on both sides of every fold.
for split_seed in range(SEED, SEED+200):
    splitter = StratifiedGroupKFold(4, shuffle=True, random_state=split_seed)
    har_folds = list(splitter.split(Xh, yh, groups))
    if all(len(np.unique(yh[tr])) == len(labels) == len(np.unique(yh[va])) for tr,va in har_folds):
        break
else:
    raise ValueError("No four-fold split covers all activities.")
print("Fold seed:", split_seed)
def scaled(model):
    return make_pipeline(VarianceThreshold(), StandardScaler(), model)
har_models = {
    "Majority": DummyClassifier(strategy="most_frequent"),
    "Logistic": scaled(LogisticRegression(max_iter=2500)),
    "Balanced logistic": scaled(LogisticRegression(max_iter=2500,class_weight="balanced")),
    "RBF SVM C=1": scaled(SVC(C=1)),
    "RBF SVM C=10": scaled(SVC(C=10)),
    "Distance-weighted neighbors": scaled(KNeighborsClassifier(n_neighbors=5,weights="distance")),
    "Random forest": RandomForestClassifier(n_estimators=240,n_jobs=WORKERS,random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(n_estimators=240,n_jobs=WORKERS,random_state=SEED)}
har_results, predictions = [], {}
for name, candidate in har_models.items():
    pred = np.empty_like(yh)
    for tr,va in har_folds:
        assert not np.intersect1d(groups[tr],groups[va]).size
        pred[va] = clone(candidate).fit(Xh.iloc[tr],yh[tr]).predict(Xh.iloc[va])
    predictions[name] = pred
    har_results.append(dict(model=name, correct=(pred == yh).sum(), accuracy=accuracy_score(yh,pred),
                            macro_f1=f1_score(yh,pred,average="macro")))
har_board = pd.DataFrame(har_results).sort_values(["correct","macro_f1"],ascending=False,kind="stable")
display(har_board)


Fold seed: 610
                         model  correct  accuracy  macro_f1
6                Random forest     1843  0.754709  0.478000
7                  Extra Trees     1806  0.739558  0.461880
4                 RBF SVM C=10     1805  0.739148  0.527703
3                  RBF SVM C=1     1738  0.711712  0.431320
1                     Logistic     1731  0.708845  0.569396
2            Balanced logistic     1694  0.693694  0.558408
5  Distance-weighted neighbors     1484  0.607699  0.422367
0                     Majority      696  0.285012  0.040327


### Sensor representations and probability blends
Append 24 principal components to the raw forest inputs to expose relationships among sensor columns. Compare the previous blend with the PCA forest, a logistic probability contribution, their combination, and a fourth Extra Trees contribution. All settings are fixed before scoring. SVM calibration keeps duplicate groups together inside each training fold.

In [3]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
def grouped_splits(rows):
    target, identities = yh[rows], groups[rows]
    for n in (3, 2):
        for seed in range(SEED, SEED+200):
            cv = StratifiedGroupKFold(n, shuffle=True, random_state=seed)
            splits = list(cv.split(Xh.iloc[rows], target, identities))
            if all(len(np.unique(target[a])) == len(labels) == len(np.unique(target[b]))
                   for a,b in splits):
                return splits
    raise ValueError("No grouped split covers every activity.")

# Raw features plus correlated sensor directions. PCA and scaling are fitted on training rows.
def sensor_forest():
    directions = make_pipeline(VarianceThreshold(), StandardScaler(),
                               PCA(n_components=24, svd_solver="randomized", random_state=SEED))
    features = FeatureUnion([("raw",FunctionTransformer()), ("directions",directions)])
    return make_pipeline(features, clone(har_models["Random forest"]))

blend_name = "Fixed forest/SVM"
variants = {blend_name:(False,False,False), "PCA forest/SVM":(True,False,False),
            "Forest/SVM/logistic":(False,True,False), "PCA forest/SVM/logistic":(True,True,False),
            "PCA forest/SVM/logistic/Extra Trees":(True,True,True)}
def activity_candidate(name, rows):
    use_pca, use_logistic, use_extra = variants[name]
    forest_model = sensor_forest() if use_pca else clone(har_models["Random forest"])
    svm = CalibratedClassifierCV(clone(har_models["RBF SVM C=10"]), method="sigmoid",
                                 cv=grouped_splits(rows),ensemble=False)
    parts = [("forest",forest_model), ("svm",svm)]
    weights = [4,2]
    if use_logistic:
        parts.append(("logistic",clone(har_models["Logistic"])))
        weights.append(1)
    if use_extra:
        parts.append(("extra",clone(har_models["Extra Trees"])))
        weights.append(2)
    return VotingClassifier(parts, voting="soft", weights=weights)

fold_number = np.zeros(len(yh), dtype=int)
for name in variants:
    pred = np.empty_like(yh)
    for fold,(tr,va) in enumerate(har_folds,1):
        pred[va] = activity_candidate(name,tr).fit(Xh.iloc[tr],yh[tr]).predict(Xh.iloc[va])
        fold_number[va] = fold
    predictions[name] = pred
    har_results.append(dict(model=name,correct=int((pred==yh).sum()),accuracy=accuracy_score(yh,pred),
                            macro_f1=f1_score(yh,pred,average="macro")))
har_board = pd.DataFrame(har_results).sort_values(["correct","macro_f1"],ascending=False,kind="stable")
display(har_board)
har_winner = har_board.iloc[0].model
print("Selected versus previous blend")
display(paired_by(yh,predictions[blend_name],predictions[har_winner],fold_number))
display(pd.DataFrame(confusion_matrix(yh,predictions[har_winner],labels=labels),index=labels,columns=labels))

                                  model  correct  accuracy  macro_f1
10                  Forest/SVM/logistic     1897  0.776822  0.607726
11              PCA forest/SVM/logistic     1894  0.775594  0.610937
12  PCA forest/SVM/logistic/Extra Trees     1887  0.772727  0.589161
8                      Fixed forest/SVM     1883  0.771089  0.571404
9                        PCA forest/SVM     1868  0.764947  0.557283
6                         Random forest     1843  0.754709  0.478000
7                           Extra Trees     1806  0.739558  0.461880
4                          RBF SVM C=10     1805  0.739148  0.527703
3                           RBF SVM C=1     1738  0.711712  0.431320
1                              Logistic     1731  0.708845  0.569396
2                     Balanced logistic     1694  0.693694  0.558408
5           Distance-weighted neighbors     1484  0.607699  0.422367
0                              Majority      696  0.285012  0.040327
Selected versus previous blend
   

## Refit and save
The bundle contains the selected pipelines, threshold and dated transaction fields used by the fraud feature builder. Treat it as private course data. Keep this notebook with the bundle. The functions below use the same feature rules for future CSV rows. Optional download is off by default.

In [7]:
output = Path("ps5_combined_models")
output.mkdir(exist_ok=True)
fraud_bundle = dict(model=fit_fraud(fraud_winner,np.arange(len(yf))), name=fraud_winner,
                    columns=cols, threshold=thresholds[fraud_winner], sklearn_version=sklearn.__version__)
history_needed = any(c not in basic for c in cols)
if history_needed:
    fraud[event_fields].to_pickle(output/"history.pkl.gz",compression="gzip")
joblib.dump(fraud_bundle,output/"fraud.joblib")
rows = np.arange(len(yh))
activity_final = activity_candidate(har_winner,rows) if har_winner in variants else clone(har_models[har_winner])
joblib.dump(activity_final.fit(Xh,yh),output/"activity.joblib")

def predict_fraud_rows(rows, folder=output):
    bundle = joblib.load(folder/"fraud.joblib")
    if any(c not in basic for c in bundle["columns"]):
        history = pd.read_pickle(folder/"history.pkl.gz")
        joined = pd.concat([history,rows[event_fields]],ignore_index=True)
        features = fraud_features(joined).iloc[len(history):]
    else:
        features = pd.DataFrame({"amount":rows.amt, "hour":rows.trans_time.str.slice(0,2).astype(int)})
    scores = fraud_scores(bundle["model"],features,bundle["columns"])
    return (scores >= bundle["threshold"]).astype(int), scores

def predict_activity_rows(predictors, folder=output):
    return joblib.load(folder/"activity.joblib").predict(predictors)

# For incoming fraud CSVs use pd.read_csv(path, dtype={k:"string" for k in entity_fields}).
# For activity CSVs use header=None and pass the first Xh.shape[1] columns.
print("Saved:", fraud_winner, "and", har_winner, "in", output.resolve())
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    files.download(shutil.make_archive(str(output),"zip",output))

Saved: Combined + context, leaf 2 and Forest/SVM/logistic in ps5_combined_models


## Search settings and metrics
Average precision (AP) summarizes precision across recall levels using each fraud model’s continuous scores. Decision thresholds affect fraud F1 and accuracy. The two objectives use separate searches. The settings and winners are printed below.

In [2]:
import importlib.util
import sys
import subprocess
if any(importlib.util.find_spec(pkg) is None for pkg in ("flaml","lightgbm","xgboost","psutil")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "flaml==2.8.0", "lightgbm", "xgboost", "psutil"])
from flaml import AutoML, tune
from sklearn.metrics import average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from importlib.metadata import version
import warnings
print({p:version(p) for p in ["flaml","lightgbm","xgboost","scikit-learn"]})

FAMILIES = ["lgbm","xgboost","rf","extra_tree"]
TRIALS, SECONDS, FRAUD_SEARCH_ROWS = 24, 90, 60000
audit = []
class FixedFolds:
    """FLAML CV adapter. Indices refer to rows in the supplied training array."""
    def __init__(self, folds):
        self.folds = folds
    def split(self, X, y=None, groups=None):
        yield from self.folds
    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.folds)

def fraud_loss(objective):
    def evaluate(X, y, estimator, *args):
        score = estimator.predict_proba(X)[:,1]
        cutoff = best_threshold(y,score,objective)
        pred = score >= cutoff
        value = accuracy_score(y,pred) if objective == "accuracy" else f1_score(y,pred,zero_division=0)
        return 1-value, {"inner_score":value}
    return evaluate

def search(X,y,inner,objective,task,window):
    """Tune on explicit training-side splits. Outer rows are absent from X and y."""
    splits = []
    for a,b in inner:
        assert len(a) and len(b) and not np.intersect1d(a,b).size
        if task == "fraud" and len(a)>FRAUD_SEARCH_ROWS:
            a,_ = train_test_split(a,train_size=FRAUD_SEARCH_ROWS,stratify=y[a],random_state=SEED)
        splits.append((np.sort(a),b))
    engine = AutoML()
    metric = fraud_loss(objective) if task == "fraud" else objective
    engine.fit(X_train=X,y_train=y,task="classification",metric=metric,
               estimator_list=FAMILIES, learner_selector="roundrobin", eval_method="cv",
               split_type=FixedFolds(splits), sample=False, auto_augment=False,
               skip_transform=True, ensemble=False, retrain_full=True, seed=SEED,
               n_jobs=WORKERS, max_iter=TRIALS, time_budget=SECONDS, verbose=0,
               custom_hp={k:{"n_estimators":{"domain":tune.lograndint(16,129),
                               "init_value":64,"low_cost_init_value":16}} for k in FAMILIES})
    audit.append(dict(task=task,window=window,objective=objective,family=engine.best_estimator,
                      families_evaluated=" / ".join(k for k,v in engine.best_config_per_estimator.items() if v is not None),
                      config=str(engine.best_config)))
    print(task, window, objective, engine.best_estimator, flush=True)
    return clone(engine.model.estimator)

{'flaml': '2.8.0', 'lightgbm': '4.7.0', 'xgboost': '3.4.1', 'scikit-learn': '1.9.1'}


## Activity recognition
Keep each outer validation group out of the full search. FLAML ranks configurations on grouped inner folds. The incumbent uses the same grouped probability calibration and fixed forest/SVM/logistic weights of 4:2:1 as before. Compare pooled predictions across all 2,442 rows.

In [4]:
forest_name = "Current forest/SVM/logistic"
har_names = [forest_name,"FLAML accuracy","FLAML macro_f1"]
predictions = {name:np.empty_like(yh) for name in har_names}
activity_fold_results = []
encoded = np.searchsorted(labels,yh)
har_arrays = Xh.to_numpy()
for fold,(tr,va) in enumerate(har_folds,1):
    assert not np.intersect1d(groups[tr],groups[va]).size
    inner = grouped_splits(tr)
    for a,b in inner:
        assert not np.intersect1d(groups[tr[a]],groups[tr[b]]).size
    for name in har_names:
        if name == forest_name:
            fitted = activity_candidate("Forest/SVM/logistic",tr).fit(Xh.iloc[tr],yh[tr])
            pred = fitted.predict(Xh.iloc[va])
        else:
            estimator = search(har_arrays[tr],encoded[tr],inner,name.split()[-1],"activity",fold)
            estimator.fit(har_arrays[tr],encoded[tr])
            pred = labels[estimator.predict(har_arrays[va]).astype(int)]
        predictions[name][va] = pred
        activity_fold_results.append(dict(fold=fold,model=name,accuracy=accuracy_score(yh[va],pred),
                                          macro_f1=f1_score(yh[va],pred,average="macro")))
har_board = pd.DataFrame([dict(model=k,correct=int((v==yh).sum()),accuracy=accuracy_score(yh,v),
                               macro_f1=f1_score(yh,v,average="macro")) for k,v in predictions.items()
                          ]).sort_values(["correct","macro_f1"],ascending=False,kind="stable")
har_winner = har_board.iloc[0].model
display(har_board)
display(pd.DataFrame(activity_fold_results))

activity 1 accuracy xgboost
activity 1 macro_f1 xgboost
activity 2 accuracy lgbm
activity 2 macro_f1 lgbm
activity 3 accuracy lgbm
activity 3 macro_f1 xgboost
activity 4 accuracy xgboost
activity 4 macro_f1 xgboost
                      model  correct  accuracy  macro_f1
             FLAML macro_f1     1919  0.785831  0.645098
             FLAML accuracy     1914  0.783784  0.614929
Current forest/SVM/logistic     1897  0.776822  0.607726
 fold                       model  accuracy  macro_f1
    1 Current forest/SVM/logistic  0.759411  0.560037
    1              FLAML accuracy  0.770867  0.570653
    1              FLAML macro_f1  0.782324  0.600029
    2 Current forest/SVM/logistic  0.790164  0.657265
    2              FLAML accuracy  0.786885  0.643438
    2              FLAML macro_f1  0.781967  0.609825
    3 Current forest/SVM/logistic  0.777049  0.586019
    3              FLAML accuracy  0.786885  0.517684
    3              FLAML macro_f1  0.772131  0.607995
    4 Current for

## Fraud detection
Apply the same strictly historical features to the current combined forest and both AutoML objectives. March–April, May–June and July–August are expanding-window development evaluations. For each, choose configuration and threshold using the two months immediately preceding that window. For the September-onward comparison, use all three development windows as the inner folds and pool their full-training-size out-of-fold scores for calibration. Each window is scored after its threshold is frozen.

In [7]:
cols = context
Xraw = Xf[cols]
# Fixed sentinel for FLAML: raw engineered features are nonnegative, including distances and ratios.
Xauto = Xraw.fillna(-1).to_numpy(dtype=np.float32)
incumbent = forest(2)
reference = "Current combined forest"
protocols = [reference+" / accuracy",reference+" / fraud_f1","FLAML / accuracy","FLAML / fraud_f1"]
dev_records, hold_records, pooled, final_specs = [], [], {k:[] for k in protocols}, {}
def calibrate(template, X, inner, y):
    truth,score = [],[]
    for a,b in inner:
        fit = clone(template).fit(X[a],y[a])
        truth.extend(y[b])
        score.extend(fit.predict_proba(X[b])[:,1])
    return np.asarray(truth),np.asarray(score)

def report(y,score,cut):
    return {**fraud_metrics(y,score,cut),"AP":average_precision_score(y,score)}

hold = np.flatnonzero(dates >= edges[-1])
before_hold = np.flatnonzero(dates < edges[-1])
windows = list(zip(edges[:-1],folds)) + [(edges[-1],(before_hold,hold))]
for i,(start,(tr,va)) in enumerate(windows):
    which = start.strftime("%Y-%m")
    if i == len(folds):
        splits = folds     # all three previous development windows
    else:
        boundary = start - pd.DateOffset(months=2)
        splits = [(tr[dates.iloc[tr].to_numpy() < boundary.to_datetime64()],
                   tr[dates.iloc[tr].to_numpy() >= boundary.to_datetime64()])]
    # Convert global indices to positional indices in X[tr] for the search.
    inner = [(np.searchsorted(tr,a),np.searchsorted(tr,b)) for a,b in splits]
    for (a,b),(ra,rb) in zip(splits,inner):
        assert np.array_equal(tr[ra],a) and np.array_equal(tr[rb],b)
        assert dates.iloc[a].max()<dates.iloc[b].min() and dates.iloc[b].max()<start
    assert dates.iloc[tr].max()<dates.iloc[va].min()
    raw = Xraw.to_numpy(dtype=np.float32)
    base_truth,base_cal = calibrate(incumbent,raw,splits,yf)
    base_score = clone(incumbent).fit(raw[tr],yf[tr]).predict_proba(raw[va])[:,1]
    for name in protocols:
        family,objective = name.split(" / ")
        if family == reference:
            template,truth,cal,score = incumbent,base_truth,base_cal,base_score
        else:
            template = search(Xauto[tr],yf[tr],inner,objective,"fraud",which)
            truth,cal = calibrate(template,Xauto,splits,yf)
            score = clone(template).fit(Xauto[tr],yf[tr]).predict_proba(Xauto[va])[:,1]
        cut = best_threshold(truth,cal,objective)
        rec = dict(window=which,model=name,threshold=cut,**report(yf[va],score,cut))
        if i < len(folds):
            dev_records.append(rec)
            pooled[name].append((yf[va],score>=cut,score))
        else:
            hold_records.append(rec)
            final_specs[name] = (template,cut)
    print("Scored",which,"on",len(va),"rows",flush=True)
dev_table = []
for name,parts in pooled.items():
    y,p,s = (np.concatenate(arr) for arr in zip(*parts))
    dev_table.append(dict(model=name,correct=int((p==y).sum()),accuracy=accuracy_score(y,p),
                          fraud_f1=f1_score(y,p),AP=average_precision_score(y,s)))
dev_board = pd.DataFrame(dev_table).sort_values(["correct","fraud_f1"],ascending=False,kind="stable")
fraud_winner = dev_board.iloc[0].model
print("Pooled forward development predictions")
display(dev_board)
display(pd.DataFrame(dev_records))
print("Later-period comparison; export protocol already determined by development performance")
display(pd.DataFrame(hold_records))
display(pd.DataFrame(audit))

fraud 2022-03 accuracy xgboost
fraud 2022-03 fraud_f1 xgboost
Scored 2022-03 on 53340 rows
fraud 2022-05 accuracy xgboost
fraud 2022-05 fraud_f1 lgbm
Scored 2022-05 on 61211 rows
fraud 2022-07 accuracy xgboost
fraud 2022-07 fraud_f1 xgboost
Scored 2022-07 on 67669 rows
fraud 2022-09 accuracy xgboost
fraud 2022-09 fraud_f1 xgboost
Scored 2022-09 on 140438 rows
Pooled forward development predictions
                             model  correct  accuracy  fraud_f1       AP
Current combined forest / fraud_f1   181888  0.998178  0.809195 0.863617
                  FLAML / fraud_f1   181876  0.998112  0.807606 0.876212
Current combined forest / accuracy   181873  0.998096  0.796719 0.863617
                  FLAML / accuracy   181710  0.997201  0.736298 0.709793
 window                              model  threshold  correct  accuracy  fraud_f1  macro_f1      auc  false_alerts  missed_frauds  caught       AP
2022-03 Current combined forest / accuracy   0.340667    53258  0.998463  0.846442  0.

## Refit and export
Export the protocol with highest pooled development accuracy, using F1 to resolve an exact tie. Freeze the fraud configuration and threshold before the later-period comparison. Refit the selected models on all supplied training rows. The private bundle includes dated history and requires the matching library versions printed above.

In [8]:
output = Path("ps5_automl_models")
output.mkdir(exist_ok=True)
template,cut = final_specs[fraud_winner]
use_auto = fraud_winner.startswith("FLAML")
final_fraud = clone(template).fit(Xauto if use_auto else Xraw.to_numpy(),yf)
joblib.dump(dict(model=final_fraud,name=fraud_winner,threshold=cut,columns=cols,fill=use_auto),output/"fraud.joblib")
fraud[event_fields].to_pickle(output/"history.pkl.gz",compression={"method":"gzip","compresslevel":1})
rows = np.arange(len(yh))
if har_winner == forest_name:
    activity_final = activity_candidate("Forest/SVM/logistic",rows).fit(Xh,yh)
else:
    winner = search(har_arrays,encoded,grouped_splits(rows),har_winner.split()[-1],"activity","full refit")
    activity_final = VotingClassifier([("winner",winner)],voting="hard").fit(Xh,yh)
joblib.dump(activity_final,output/"activity.joblib")
def predict_fraud_rows(rows,folder=output):
    bundle = joblib.load(folder/"fraud.joblib")
    history = pd.read_pickle(folder/"history.pkl.gz")
    x = fraud_features(pd.concat([history,rows[event_fields]],ignore_index=True)).iloc[len(history):][bundle["columns"]]
    if bundle["fill"]:
        x = x.fillna(-1)
    scores = bundle["model"].predict_proba(x.to_numpy(dtype=np.float32))[:,1]
    return (scores>=bundle["threshold"]).astype(int),scores
def predict_activity_rows(rows,folder=output):
    return joblib.load(folder/"activity.joblib").predict(rows)
print("Exported",fraud_winner,"and",har_winner)
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    files.download(shutil.make_archive(str(output),"zip",output))

activity full refit macro_f1 lgbm
Exported Current combined forest / fraud_f1 and FLAML macro_f1
